# Kaggle — CellViT nucleus feature extraction

Run the preflight first. Full extraction starts only after the exact checkpoint, postprocessor, DINO forward, paths, GPU, and writable disk have passed. Internet must be enabled unless `DINO_MODEL` points to a mounted local model directory.

In [ ]:
from pathlib import Path
import os, subprocess, sys

CODAPATH = Path('/kaggle/working/codapath')
if not CODAPATH.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/CryAndRRich/codapath.git', str(CODAPATH)])
os.chdir(CODAPATH)
print('repo:', CODAPATH)

In [ ]:
# Do not install CellViT's full dependency tree over Kaggle's PyTorch.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--require-hashes', '-r', 'requirements-kaggle-cellvit.txt'])
import importlib.util
needed = {'transformers': 'transformers>=4.27', 'yaml': 'PyYAML>=6.0', 'einops': 'einops>=0.6.1'}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

# These scientific packages are normally preinstalled on Kaggle. Preflight
# will report the exact missing/ABI-broken import instead of silently changing them.
print('dependency setup complete')

In [ ]:
# ---- EDIT ONLY THIS CELL ----
DATASET = 'pathmnist'  # pathmnist | skintissue; HistoSet needs per-source MPP
SEED = 42
DATA_PATH = '/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz'
CHECKPOINT_PATH = '/kaggle/input/cellvit-checkpoints/CellViT-256-x20.pth'
CACHE_DIR = '/kaggle/working/nucleus_features'
DINO_MODEL = 'facebook/dinov2-base'  # or a mounted local model directory
BATCH_SIZE = 2
DINO_CROP_BATCH_SIZE = 32
OVERWRITE = False

assert Path(DATA_PATH).exists(), DATA_PATH
assert Path(CHECKPOINT_PATH).is_file(), CHECKPOINT_PATH
assert not str(CACHE_DIR).startswith('/kaggle/input/'), 'Kaggle input is read-only'

In [ ]:
# Exact one-batch integration test. Do not continue if this cell fails.
preflight = [
    sys.executable, 'scripts/preflight_nucleus_kaggle.py',
    '--dataset', DATASET, '--data_path', DATA_PATH,
    '--checkpoint', CHECKPOINT_PATH, '--cache_dir', CACHE_DIR,
    '--vit_name', DINO_MODEL, '--seed', str(SEED), '--smoke_samples', '2',
]
subprocess.check_call(preflight)

In [ ]:
command = [
    sys.executable, 'scripts/extract_nucleus_features.py',
    '--dataset', DATASET, '--data_path', DATA_PATH,
    '--checkpoint', CHECKPOINT_PATH, '--cache_dir', CACHE_DIR,
    '--vit_name', DINO_MODEL, '--seed', str(SEED), '--device', 'cuda',
    '--batch_size', str(BATCH_SIZE),
    '--dino_crop_batch_size', str(DINO_CROP_BATCH_SIZE),
]
if OVERWRITE:
    command.append('--overwrite')
subprocess.check_call(command)

In [ ]:
# Validate the finished artifact before saving it as a Kaggle output dataset.
import json
from load_data import get_data_loaders, get_sample_ids
from nucleus.cache import load_nucleus_cache
cache_path = Path(CACHE_DIR) / f'{DATASET}_seed{SEED}'
manifest = json.loads((cache_path / 'manifest.json').read_text())
required = ['offsets.npy', 'confidence.npy', 'sample_ids.npy',
            'cellvit_embeddings.npy', 'cell_dino_features.npy', 'manifest.json']
missing = [name for name in required if not (cache_path / name).exists()]
assert not missing, missing
train_loader, _, _ = get_data_loaders(DATA_PATH, SEED, verbose=True)
cache = load_nucleus_cache(cache_path, expected_sample_ids=get_sample_ids(train_loader.dataset))
assert cache.num_patches == len(train_loader.dataset)
print(json.dumps(manifest, indent=2))
print('cache size GiB:', sum(p.stat().st_size for p in cache_path.rglob('*') if p.is_file()) / 2**30)

In [ ]:
# Visual QC: red boundaries must follow nuclei, not background/whole glands.
from PIL import Image
from IPython.display import display
for path in sorted((cache_path / 'qc').glob('*.png'))[:8]:
    display(Image.open(path))